In [0]:
%sql
-- Monthly Consolidated Revenue - Total revenue across both companies by month with month-over-month growth percentage
WITH monthly_revenue AS (
  SELECT 
    dc.company_name,
    udc.year_month,
    SUM(CASE 
      WHEN udc.account_category = 'Operating Revenue' AND da.account_type = 'Revenue' THEN udc.credit_amount 
      WHEN udc.account_category = 'Operating Revenue' AND da.account_type = 'Contra Revenue' THEN -udc.debit_amount
      ELSE 0 
    END) AS revenue
  FROM 03_global_tech_gold.03_data_cube.unified_data_cube udc
  LEFT JOIN 03_global_tech_gold.01_dims_tables.dim_company dc ON udc.company_key = dc.company_key
  LEFT JOIN 03_global_tech_gold.01_dims_tables.dim_account da ON udc.account_key = da.account_key
  WHERE udc.account_category = 'Operating Revenue'
    AND udc.gl_status = 'Posted'
  GROUP BY dc.company_name, udc.year_month
),
consolidated_revenue AS (
  SELECT 
    year_month,
    SUM(revenue) AS total_revenue
  FROM monthly_revenue
  GROUP BY year_month
)
SELECT 
  year_month,
  ROUND(total_revenue, 2) AS total_revenue,
  ROUND(LAG(total_revenue) OVER (ORDER BY year_month), 2) AS prev_month_revenue,
  ROUND(
    ((total_revenue - LAG(total_revenue) OVER (ORDER BY year_month)) / 
    NULLIF(LAG(total_revenue) OVER (ORDER BY year_month), 0)) * 100, 2
  ) AS mom_growth_pct
FROM consolidated_revenue
ORDER BY year_month

In [0]:
%sql
-- Cost of Sales by Month - Direct costs incurred to deliver products and services, tracked monthly by company
SELECT 
  dc.company_name,
  udc.year_month,
  ROUND(SUM(CASE 
    WHEN udc.account_category = 'Direct Costs' AND da.account_type = 'COGS' THEN udc.debit_amount 
    WHEN udc.account_category = 'Direct Costs' AND da.account_type = 'Contra COGS' THEN -udc.credit_amount
    ELSE 0 
  END), 2) AS cost_of_sales
FROM 03_global_tech_gold.03_data_cube.unified_data_cube udc
LEFT JOIN 03_global_tech_gold.01_dims_tables.dim_company dc ON udc.company_key = dc.company_key
LEFT JOIN 03_global_tech_gold.01_dims_tables.dim_account da ON udc.account_key = da.account_key
WHERE udc.account_category = 'Direct Costs'
  AND udc.gl_status = 'Posted'
GROUP BY dc.company_name, udc.year_month
ORDER BY udc.year_month, dc.company_name

In [0]:
%sql
-- Monthly Gross Profit Margin - Profitability percentage after deducting cost of sales from revenue
WITH monthly_metrics AS (
  SELECT 
    dc.company_name,
    udc.year_month,
    SUM(CASE 
      WHEN udc.account_category = 'Operating Revenue' AND da.account_type = 'Revenue' THEN udc.credit_amount 
      WHEN udc.account_category = 'Operating Revenue' AND da.account_type = 'Contra Revenue' THEN -udc.debit_amount
      ELSE 0 
    END) AS revenue,
    SUM(CASE 
      WHEN udc.account_category = 'Direct Costs' AND da.account_type = 'COGS' THEN udc.debit_amount 
      WHEN udc.account_category = 'Direct Costs' AND da.account_type = 'Contra COGS' THEN -udc.credit_amount
      ELSE 0 
    END) AS cogs
  FROM 03_global_tech_gold.03_data_cube.unified_data_cube udc
  LEFT JOIN 03_global_tech_gold.01_dims_tables.dim_company dc ON udc.company_key = dc.company_key
  LEFT JOIN 03_global_tech_gold.01_dims_tables.dim_account da ON udc.account_key = da.account_key
  WHERE udc.account_category IN ('Operating Revenue', 'Direct Costs')
    AND udc.gl_status = 'Posted'
  GROUP BY dc.company_name, udc.year_month
)
SELECT 
  company_name,
  year_month,
  ROUND(revenue, 2) AS revenue,
  ROUND(cogs, 2) AS cogs,
  ROUND(revenue - cogs, 2) AS gross_profit,
  ROUND(((revenue - cogs) / NULLIF(revenue, 0)) * 100, 2) AS gross_profit_margin_pct
FROM monthly_metrics
WHERE revenue > 0
ORDER BY year_month, company_name

In [0]:
%sql
-- Operating Expense Breakdown - Categorization of overhead costs by account range and company
SELECT 
  dc.company_name,
  da.account_name,
  da.account_code,
  udc.year_month,
  ROUND(SUM(udc.debit_amount), 2) AS total_expense
FROM 03_global_tech_gold.03_data_cube.unified_data_cube udc
LEFT JOIN 03_global_tech_gold.01_dims_tables.dim_company dc ON udc.company_key = dc.company_key
LEFT JOIN 03_global_tech_gold.01_dims_tables.dim_account da ON udc.account_key = da.account_key
WHERE udc.account_category = 'Operating Expense'
  AND udc.gl_status = 'Posted'
  AND da.account_type = 'Expense'
GROUP BY dc.company_name, da.account_name, da.account_code, udc.year_month
ORDER BY udc.year_month, dc.company_name, total_expense DESC

In [0]:
%sql
-- Average Compensation by Position - Mean total compensation (salary, bonus, overtime, commission) grouped by position level and company
SELECT 
  dc.company_name,
  de.position,
  COUNT(DISTINCT udc.employee_key) AS employee_count,
  ROUND(AVG(udc.total_compensation), 2) AS avg_total_compensation,
  ROUND(AVG(udc.allowances), 2) AS avg_base_salary,
  ROUND(AVG(udc.bonus), 2) AS avg_bonus,
  ROUND(AVG(udc.overtime_pay), 2) AS avg_overtime,
  ROUND(AVG(udc.commission), 2) AS avg_commission
FROM 03_global_tech_gold.03_data_cube.unified_data_cube udc
LEFT JOIN 03_global_tech_gold.01_dims_tables.dim_company dc ON udc.company_key = dc.company_key
LEFT JOIN 03_global_tech_gold.01_dims_tables.dim_employee de ON udc.employee_key = de.employee_key
WHERE udc.total_compensation IS NOT NULL
  AND udc.payroll_status = 'Paid'
GROUP BY dc.company_name, de.position
ORDER BY dc.company_name, avg_total_compensation DESC

In [0]:
%sql
-- Net Profit by Month - Bottom-line profitability after all costs including COGS, operating expenses, etc.
WITH monthly_pl AS (
  SELECT 
    dc.company_name,
    udc.year_month,
    SUM(CASE 
      WHEN da.account_type IN ('Revenue') THEN udc.credit_amount 
      WHEN da.account_type IN ('Contra Revenue') THEN -udc.debit_amount
      ELSE 0 
    END) AS total_revenue,
    SUM(CASE 
      WHEN da.account_type IN ('COGS') THEN udc.debit_amount 
      WHEN da.account_type IN ('Contra COGS') THEN -udc.credit_amount
      ELSE 0 
    END) AS total_cogs,
    SUM(CASE 
      WHEN da.account_type = 'Expense' THEN udc.debit_amount 
      ELSE 0 
    END) AS total_expenses
  FROM 03_global_tech_gold.03_data_cube.unified_data_cube udc
  LEFT JOIN 03_global_tech_gold.01_dims_tables.dim_company dc ON udc.company_key = dc.company_key
  LEFT JOIN 03_global_tech_gold.01_dims_tables.dim_account da ON udc.account_key = da.account_key
  WHERE udc.gl_status = 'Posted'
  GROUP BY dc.company_name, udc.year_month
)
SELECT 
  company_name,
  year_month,
  ROUND(total_revenue, 2) AS total_revenue,
  ROUND(total_cogs, 2) AS total_cogs,
  ROUND(total_expenses, 2) AS total_expenses,
  ROUND(total_revenue - total_cogs, 2) AS gross_profit,
  ROUND(total_revenue - total_cogs - total_expenses, 2) AS net_profit,
  ROUND(((total_revenue - total_cogs - total_expenses) / NULLIF(total_revenue, 0)) * 100, 2) AS net_profit_margin_pct
FROM monthly_pl
WHERE total_revenue > 0
ORDER BY year_month, company_name

In [0]:
%sql
-- Overtime and Bonus Analysis - Variable compensation spending by department with overtime-to-base salary ratio
SELECT 
  dc.company_name,
  dd.department_name,
  udc.year_month,
  ROUND(SUM(udc.allowances), 2) AS total_base_salary,
  ROUND(SUM(udc.overtime_pay), 2) AS total_overtime,
  ROUND(SUM(udc.bonus), 2) AS total_bonus,
  ROUND(SUM(udc.commission), 2) AS total_commission,
  ROUND(SUM(udc.overtime_pay + udc.bonus + udc.commission), 2) AS total_variable_comp,
  ROUND((SUM(udc.overtime_pay) / NULLIF(SUM(udc.allowances), 0)) * 100, 2) AS overtime_to_base_ratio_pct,
  ROUND((SUM(udc.overtime_pay + udc.bonus + udc.commission) / NULLIF(SUM(udc.allowances), 0)) * 100, 2) AS variable_to_base_ratio_pct
FROM 03_global_tech_gold.03_data_cube.unified_data_cube udc
LEFT JOIN 03_global_tech_gold.01_dims_tables.dim_company dc ON udc.company_key = dc.company_key
LEFT JOIN 03_global_tech_gold.01_dims_tables.dim_department dd ON udc.department_key = dd.department_key
WHERE udc.total_compensation IS NOT NULL
  AND udc.payroll_status = 'Paid'
GROUP BY dc.company_name, dd.department_name, udc.year_month
ORDER BY udc.year_month, dc.company_name, total_variable_comp DESC

In [0]:
%sql
-- Cost per Department - Total departmental cost combining payroll expenses
SELECT 
  dc.company_name,
  dd.department_name,
  dd.cost_center,
  udc.year_month,
  ROUND(SUM(udc.total_compensation), 2) AS total_payroll_cost,
  ROUND(SUM(udc.allowances), 2) AS base_salary_cost,
  ROUND(SUM(udc.bonus + udc.overtime_pay + udc.commission), 2) AS variable_comp_cost,
  COUNT(DISTINCT udc.employee_key) AS employee_count,
  ROUND(SUM(udc.total_compensation) / COUNT(DISTINCT udc.employee_key), 2) AS avg_cost_per_employee
FROM 03_global_tech_gold.03_data_cube.unified_data_cube udc
LEFT JOIN 03_global_tech_gold.01_dims_tables.dim_company dc ON udc.company_key = dc.company_key
LEFT JOIN 03_global_tech_gold.01_dims_tables.dim_department dd ON udc.department_key = dd.department_key
WHERE udc.total_compensation IS NOT NULL
  AND udc.payroll_status = 'Paid'
GROUP BY dc.company_name, dd.department_name, dd.cost_center, udc.year_month
ORDER BY udc.year_month, dc.company_name, total_payroll_cost DESC

In [0]:
%sql
-- Headcount Distribution by Department - Active employee count by department
SELECT 
  dc.company_name,
  dd.department_name,
  udc.year_month,
  COUNT(DISTINCT udc.employee_key) AS headcount,
  ROUND(COUNT(DISTINCT udc.employee_key) * 100.0 / 
    SUM(COUNT(DISTINCT udc.employee_key)) OVER (PARTITION BY dc.company_name, udc.year_month), 2
  ) AS pct_of_company_headcount
FROM 03_global_tech_gold.03_data_cube.unified_data_cube udc
LEFT JOIN 03_global_tech_gold.01_dims_tables.dim_company dc ON udc.company_key = dc.company_key
LEFT JOIN 03_global_tech_gold.01_dims_tables.dim_department dd ON udc.department_key = dd.department_key
WHERE udc.payroll_status = 'Paid'
  AND udc.employee_key IS NOT NULL
GROUP BY dc.company_name, dd.department_name, udc.year_month
ORDER BY udc.year_month, dc.company_name, headcount DESC

In [0]:
%sql
-- Payroll Cost as % of Company Revenue - Labor efficiency ratio
WITH monthly_payroll AS (
  SELECT 
    company_key,
    year_month,
    SUM(total_compensation) AS total_payroll_cost
  FROM 03_global_tech_gold.03_data_cube.unified_data_cube
  WHERE total_compensation IS NOT NULL
    AND payroll_status = 'Paid'
  GROUP BY company_key, year_month
),
monthly_revenue AS (
  SELECT 
    udc.company_key,
    udc.year_month,
    SUM(CASE 
      WHEN da.account_type = 'Revenue' THEN udc.credit_amount 
      WHEN da.account_type = 'Contra Revenue' THEN -udc.debit_amount
      ELSE 0 
    END) AS total_revenue
  FROM 03_global_tech_gold.03_data_cube.unified_data_cube udc
  LEFT JOIN 03_global_tech_gold.01_dims_tables.dim_account da ON udc.account_key = da.account_key
  WHERE udc.account_category = 'Operating Revenue'
    AND udc.gl_status = 'Posted'
  GROUP BY udc.company_key, udc.year_month
)
SELECT 
  dc.company_name,
  mp.year_month,
  ROUND(mp.total_payroll_cost, 2) AS total_payroll_cost,
  ROUND(mr.total_revenue, 2) AS total_revenue,
  ROUND((mp.total_payroll_cost / NULLIF(mr.total_revenue, 0)) * 100, 2) AS payroll_as_pct_of_revenue
FROM monthly_payroll mp
INNER JOIN monthly_revenue mr 
  ON mp.company_key = mr.company_key 
  AND mp.year_month = mr.year_month
LEFT JOIN 03_global_tech_gold.01_dims_tables.dim_company dc 
  ON mp.company_key = dc.company_key
WHERE mr.total_revenue > 0
ORDER BY mp.year_month, dc.company_name